In [1]:
!pip install easyfsl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 1.5 MB/s eta 0:00:00


In [2]:
!pip -q install gdown

import tarfile
import subprocess
from pathlib import Path

DRIVE_FOLDER = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"

DOWNLOAD_DIR = Path("/kaggle/working/miniimagenet_archives")
DATA_DIR = Path("/kaggle/working/miniimagenet")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the files in the shared Drive folder
result = subprocess.run([
    "gdown", "--folder", DRIVE_FOLDER,
    "-O", str(DOWNLOAD_DIR)
], check=False)

print("Download command exit code:", result.returncode)
print("Downloaded files:")
for p in DOWNLOAD_DIR.rglob("*"):
    if p.is_file():
        print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")

# Find and extract train, val, and test archives
for split in ("train", "val", "test"):
    matches = list(DOWNLOAD_DIR.rglob(f"{split}.tar"))
    if not matches:
        raise FileNotFoundError(
            f"{split}.tar was not found. Check that the Drive folder "
            "is publicly accessible and the download completed."
        )

    archive = matches[0]
    out_dir = DATA_DIR / split
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting {archive.name} into {out_dir} ...")
    with tarfile.open(archive, "r:*") as tar:
        tar.extractall(out_dir, filter="data")

    print(f"Finished extracting {split}")

print("\nTop-level extracted structure:")
for split_dir in sorted(DATA_DIR.iterdir()):
    if split_dir.is_dir():
        print(split_dir.name, "->", [
            p.name for p in list(split_dir.iterdir())[:8]
        ])

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v
From (redirected): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v&confirm=t&uuid=c5dd4e6b-ded2-4bbb-99b1-07cba664351c
To: /kaggle/working/miniimagenet_archives/test.tar
  0%|          | 0.00/39.2M [00:00<?, ?B/s]

Processing file 1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v test.tar
Processing file 107FTosYIeBn5QbynR46YG91nHcJ70whs train.tar
Processing file 1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl val.tar


100%|██████████| 39.2M/39.2M [00:00<00:00, 83.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs
From (redirected): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs&confirm=t&uuid=283d4c47-98f3-4370-81d1-d529569d6d42
To: /kaggle/working/miniimagenet_archives/train.tar
100%|██████████| 126M/126M [00:01<00:00, 76.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl
To: /kaggle/working/miniimagenet_archives/val.tar
100%|██████████| 30.8M/30.8M [00:00<00:00, 107MB/s] 
Download completed


Download command exit code: 0
Downloaded files:
val.tar: 30.8 MB
train.tar: 125.9 MB
test.tar: 39.2 MB

Extracting train.tar into /kaggle/working/miniimagenet/train ...
Finished extracting train

Extracting val.tar into /kaggle/working/miniimagenet/val ...
Finished extracting val

Extracting test.tar into /kaggle/working/miniimagenet/test ...
Finished extracting test

Top-level extracted structure:
test -> ['test']
train -> ['train']
val -> ['val']


In [3]:
# Environment and imports
# Run this notebook in Kaggle with the MiniImageNet class-folder dataset attached.
import os, time, random
from collections import defaultdict
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

ROOT_DIRECTORY = "/kaggle/working/miniimagenet"
TRAIN_DIR = f"{ROOT_DIRECTORY}/train/train"
VAL_DIR   = f"{ROOT_DIRECTORY}/val/val"
TEST_DIR  = f"{ROOT_DIRECTORY}/test/test"

SAVE_PATH = '/kaggle/working/saved_models'
os.makedirs(SAVE_PATH, exist_ok=True)

# ---------------- Experiment configuration (single source of truth) ----------------
IMAGE_SIZE = 84
MAX_EPOCHS = 10
N_TRAINING_EPISODES = 360
N_VALIDATION_TASKS = 600
N_TEST_TASKS = 600
TRAIN_WAY = 30
TEST_WAY = 5
SHOT = 5
QUERY = 15
# Speedup: Process full support (150) and query (450) batches without tiny chunk loops
FEATURE_CHUNK_SIZE = 600
ATTENTION_CHUNK_SIZE = 600

# Novelty configuration: choose 'euclidean', 'cosine', or 'hybrid'
DISTANCE_METRIC = 'cosine'   # <-- Your novelty!
COSINE_TEMP = 10.0           # Temperature scaling so CrossEntropy logits are sharp
USE_AMP = True               # Uses Tesla T4 Tensor Cores for 2x faster training

# Multi-scale stages (ResNet-18 module names; the last node of each module is used).
# The number of stages is derived from this tuple, and STAGE_WEIGHTS must match its length.
STAGE_NODES   = ('maxpool', 'layer1', 'layer2', 'layer3', 'layer4')
STAGE_WEIGHTS = (1.0, 1.1, 1.2, 1.3, 1.4)

# NOTE: miniImageNet is a subset of ImageNet-1k, so ImageNet-pretrained weights have already
# seen the val/test classes. Set to False for a leakage-free few-shot benchmark.
USE_IMAGENET_PRETRAINED = True
NUM_WORKERS = 2   # raise to 2-4 on Kaggle if data loading is the bottleneck

# ImageNet normalization matches the pretrained ResNet-18 weights.
_norm = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

# Resize makes IMAGE_SIZE actually take effect (previously it was only used for a dummy tensor).
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    _norm,
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    _norm,
])
print('Imports and configuration ready.')

Device: cuda
GPU: Tesla T4
Imports and configuration ready.


In [4]:
from pathlib import Path

for split, path in [
    ("train", TRAIN_DIR),
    ("val", VAL_DIR),
    ("test", TEST_DIR),
]:
    p = Path(path)
    classes = [d for d in p.iterdir() if d.is_dir()] if p.exists() else []

    print(f"{split}: exists={p.exists()}, class folders={len(classes)}")
    print("Examples:", [d.name for d in classes[:5]])

train: exists=True, class folders=64
Examples: ['n03220513', 'n01532829', 'n03400231', 'n04243546', 'n02795169']
val: exists=True, class folders=16
Examples: ['n02174001', 'n03770439', 'n03075370', 'n03980874', 'n02971356']
test: exists=True, class folders=20
Examples: ['n03272010', 'n02099601', 'n04146614', 'n02129165', 'n03775546']


In [5]:
# Evaluation helpers
@torch.no_grad()
def evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels):
    scores = model(support_images.to(device), support_labels.to(device), query_images.to(device))
    predictions = scores.argmax(dim=1)
    labels = query_labels.to(device)
    return (predictions == labels).sum().item(), labels.numel()

@torch.no_grad()
def evaluate(data_loader, model, desc='Evaluating'):
    model.eval()
    correct_total, prediction_total, n_tasks = 0, 0, 0
    for support_images, support_labels, query_images, query_labels, _ in tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
        correct, total = evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels)
        correct_total += correct
        prediction_total += total
        n_tasks += 1
    if prediction_total == 0:
        raise RuntimeError('Evaluation loader produced no query examples.')
    if n_tasks != len(data_loader):
        raise RuntimeError(f'Evaluated {n_tasks} tasks but loader promised {len(data_loader)}.')
    return 100.0 * correct_total / prediction_total

In [6]:
# Multi-scale attention model
class Self_Attn(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f'in_dim must be >= 8, got {in_dim}')
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)  # B,N,C/8
        k = self.key_conv(x).reshape(b, -1, n)                   # B,C/8,N
        attention = torch.softmax(torch.bmm(q, k), dim=-1)        # B,N,N
        v = self.value_conv(x).reshape(b, c, n)                   # B,C,N
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x

class PrototypicalNetworks(nn.Module):
    def __init__(self, backbone, image_size, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=False, gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 distance_metric='cosine', cosine_temp=10.0):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f'{len(weight)} weights given for {len(stage_nodes)} stages.')
        self.weightlearnable = weightlearnable
        self.gammalearnable = gammalearnable
        self.feature_chunk_size = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages = len(stage_nodes)
        self.distance_metric = distance_metric
        
        # Learnable temperature scalar for cosine distance scaling
        self.cosine_temp = nn.Parameter(torch.tensor(float(cosine_temp)), requires_grad=True)

        self.stage_weights = nn.Parameter(torch.tensor(list(weight), dtype=torch.float32),
                                          requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + '.')]
            if not matches:
                raise ValueError(f'Stage {name!r} not found in backbone graph.')
            return_nodes[matches[-1]] = f'stage_{i}'
        print('Stage nodes:', return_nodes)

        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f'stage_{i}'].shape[1] for i in range(self.n_stages)]
        print('Selected feature channels:', channels,
              '| spatial sizes:', [tuple(dummy[f'stage_{i}'].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

    def _extract_in_chunks(self, images):
        # Fast path: if batch fits in chunk_size, skip Python loop & dict rebuilding!
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {key: [value] for key, value in feats.items()}
            else:
                for key, value in feats.items(): collected[key].append(value)
        return {key: torch.cat(values, dim=0) for key, values in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        # Fast path: skip chunking loop when batch size <= attention_chunk_size
        if features.size(0) <= self.attention_chunk_size:
            return self.global_avg_pool(module(features, gamma)).flatten(1)
        pooled = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
        return torch.cat(pooled, dim=0)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query = self._attend_and_pool(query_features, stage_idx, self.gamma2_l)
        
        # Vectorized prototype computation (faster than Python list comprehension over 30 labels)
        labels = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == label].mean(0) for label in labels])

        if self.distance_metric == 'euclidean':
            return torch.cdist(z_query, prototypes, p=2)
            
        elif self.distance_metric == 'cosine':
            # L2-normalize query and prototype vectors onto unit hypersphere
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            # Cosine distance = (1 - cosine_similarity) scaled by temperature
            cos_sim = torch.mm(z_q_norm, proto_norm.t())
            return self.cosine_temp * (1.0 - cos_sim)
            
        elif self.distance_metric == 'hybrid':
            # Combines L2-normalized Euclidean distance and Cosine distance
            euc_dist = torch.cdist(z_query, prototypes, p=2)
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            cos_dist = self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
            return 0.5 * euc_dist + 0.5 * cos_dist
        else:
            raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats = self._extract_in_chunks(query_images)
        distances = torch.stack([
            self._stage_distance(support_feats[f'stage_{i}'], query_feats[f'stage_{i}'], support_labels, i)
            for i in range(self.n_stages)
        ])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print('Model class defined.')

Model class defined.


In [7]:
# Fast episode sampler compatibility patch for EasyFSL / newer PyTorch
# The patch uses precomputed labels; it never loads image files to discover labels.
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(f'Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.')
    required = n_shot + n_query
    too_small = [label for label, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f'Classes have fewer than {required} images: {too_small[:5]}')

TaskSampler.__init__ = _patched_task_sampler_init
print('TaskSampler patched; labels are read from metadata only.')

TaskSampler patched; labels are read from metadata only.


In [8]:
# Dataset setup (must run BEFORE the training cell, after the sampler patch).
train_set = ImageFolder(TRAIN_DIR, transform=train_transform)
val_set   = ImageFolder(VAL_DIR, transform=eval_transform)
test_set  = ImageFolder(TEST_DIR, transform=eval_transform)

# EasyFSL sampler compatibility.
def _get_targets(dataset):
    return dataset.targets

import types
train_set.get_labels = types.MethodType(_get_targets, train_set)
val_set.get_labels   = types.MethodType(_get_targets, val_set)
test_set.get_labels  = types.MethodType(_get_targets, test_set)

print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

assert len(train_set.classes) >= TRAIN_WAY, "Not enough training classes for TRAIN_WAY."
assert len(val_set.classes) >= TEST_WAY, "Not enough validation classes for TEST_WAY."
assert len(test_set.classes) >= TEST_WAY, "Not enough test classes for TEST_WAY."

def make_loader(dataset, n_way, n_tasks):
    sampler = TaskSampler(dataset, n_way=n_way, n_shot=SHOT, n_query=QUERY, n_tasks=n_tasks)
    loader = DataLoader(
        dataset, batch_sampler=sampler, num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"), collate_fn=sampler.episodic_collate_fn,
    )
    # Fail loudly if the loader is not exactly as configured (e.g. stale cell / old value).
    assert len(loader) == n_tasks, f"Loader has {len(loader)} tasks, expected {n_tasks}."
    return loader

# Always rebuilt from the config in the config cell -> re-running this cell picks up new values.
train_loader = make_loader(train_set, TRAIN_WAY, N_TRAINING_EPISODES)
val_loader   = make_loader(val_set,   TEST_WAY,  N_VALIDATION_TASKS)
test_loader  = make_loader(test_set,  TEST_WAY,  N_TEST_TASKS)
print(f"Episodes per epoch: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

# Smoke test: verify the episode shapes really follow the configuration.
def check_episode(loader, way):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.shape == (way * SHOT, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(s_lab) == way * SHOT and len(q_lab) == way * QUERY
    assert len(set(s_lab.tolist())) == way
    print(f"{way}-way episode OK:", tuple(s_img.shape), tuple(q_img.shape))

t0 = time.time()
check_episode(train_loader, TRAIN_WAY)
check_episode(val_loader, TEST_WAY)
print(f"Smoke test done in {time.time() - t0:.2f}s")

Classes: train=64, val=16, test=20
Images: train=38400, val=9600, test=12000
Episodes per epoch: train=360, val=600, test=600
30-way episode OK: (150, 3, 84, 84) (450, 3, 84, 84)
5-way episode OK: (25, 3, 84, 84) (75, 3, 84, 84)
Smoke test done in 1.87s


In [9]:
# Train and validate; save the best checkpoint.
gamma_init = 0.2
weights_lr = 0.005
backbone_lr = 1e-4

print('Initializing ResNet-18...')
backbone = None
if USE_IMAGENET_PRETRAINED:
    try:
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
    except Exception as exc:
        print('Could not load torchvision pretrained weights; using random initialization:', exc)
if backbone is None:
    backbone = resnet18(weights=None)
backbone.fc = nn.Identity()

# Integrated new parameters: distance_metric and cosine_temp
model = PrototypicalNetworks(
    backbone=backbone, image_size=IMAGE_SIZE, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
    weightlearnable=True, gammalearnable=False, gamma1=gamma_init, gamma2=gamma_init,
    feature_chunk_size=FEATURE_CHUNK_SIZE, attention_chunk_size=ATTENTION_CHUNK_SIZE,
    distance_metric=DISTANCE_METRIC, cosine_temp=COSINE_TEMP,
).to(device)

# Updated special parameters list to include model.cosine_temp
special = [model.stage_weights, model.gamma1_l, model.gamma2_l, model.cosine_temp]
special_ids = {id(p) for p in special}
regular_params = [p for p in model.parameters() if id(p) not in special_ids]

# Reconstructed param_groups with specific learning rate for cosine_temp
param_groups = [
    {'params': regular_params, 'lr': backbone_lr},
    {'params': [model.cosine_temp], 'lr': weights_lr},
]
if model.weightlearnable:
    param_groups.append({'params': [model.stage_weights], 'lr': weights_lr})
if model.gammalearnable:
    param_groups.append({'params': [model.gamma1_l, model.gamma2_l], 'lr': weights_lr})

optimizer = optim.Adam(param_groups)

# Added Learning Rate Scheduler and GradScaler for AMP
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
trainable_params = [p for p in model.parameters() if p.requires_grad]

criterion = nn.CrossEntropyLoss()
best_val_acc = -1.0
best_checkpoint_path = os.path.join(SAVE_PATH, f'best_model_{SHOT}shot.pth')

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    loss_sum, n_episodes = 0.0, 0
    start = time.time()
    progress = tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch}/{MAX_EPOCHS}')
    
    for support_images, support_labels, query_images, query_labels, _ in progress:
        optimizer.zero_grad(set_to_none=True)
        
        # Wrapped forward pass and loss computation in autocast
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            scores = model(support_images.to(device, non_blocking=True), support_labels.to(device),
                           query_images.to(device, non_blocking=True))
            loss = criterion(scores, query_labels.to(device))
            
        # Updated backward and step sequences to use the gradient scaler
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        
        loss_sum += loss.item(); n_episodes += 1
        progress.set_postfix(loss=f'{loss.item():.4f}')
        
    if n_episodes != N_TRAINING_EPISODES:
        raise RuntimeError(f'Ran {n_episodes} episodes, config says {N_TRAINING_EPISODES}. '
                           'Re-run the dataset cell to rebuild the loaders.')
    
    # Step the learning rate scheduler at the end of every training epoch
    scheduler.step()
    
    train_loss = loss_sum / n_episodes
    val_acc = evaluate(val_loader, model, desc=f'Validating epoch {epoch}')
    print(f'Epoch {epoch}: train loss={train_loss:.4f}, val accuracy={val_acc:.2f}%, time={(time.time()-start)/60:.1f} min')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_accuracy': val_acc,
                    'config': {'train_way': TRAIN_WAY, 'test_way': TEST_WAY, 'shot': SHOT, 'query': QUERY,
                               'image_size': IMAGE_SIZE, 'stage_nodes': list(STAGE_NODES)}},
                   best_checkpoint_path)
        print('Saved best checkpoint:', best_checkpoint_path)

print(f'Best validation accuracy: {best_val_acc:.2f}%')


Initializing ResNet-18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Stage nodes: {'maxpool': 'stage_0', 'layer1.1.relu_1': 'stage_1', 'layer2.1.relu_1': 'stage_2', 'layer3.1.relu_1': 'stage_3', 'layer4.1.relu_1': 'stage_4'}
Selected feature channels: [64, 64, 128, 256, 512] | spatial sizes: [(21, 21), (21, 21), (11, 11), (6, 6), (3, 3)]


Epoch 1/10:   0%|          | 0/360 [00:00<?, ?it/s]

Validating epoch 1:   0%|          | 0/600 [00:00<?, ?it/s]

Epoch 1: train loss=0.5270, val accuracy=83.07%, time=2.9 min
Saved best checkpoint: /kaggle/working/saved_models/best_model_5shot.pth


... (30 outputs left out)

Epoch 9: train loss=0.0011, val accuracy=79.98%, time=2.9 min


Epoch 10/10:   0%|          | 0/360 [00:00<?, ?it/s]

Validating epoch 10:   0%|          | 0/600 [00:00<?, ?it/s]

Epoch 10: train loss=0.0010, val accuracy=79.09%, time=2.9 min
Best validation accuracy: 83.07%


In [10]:
# Final evaluation on held-out test classes
if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    state = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state)
    print(f"Loaded best checkpoint (epoch {checkpoint.get('epoch', '?') if isinstance(checkpoint, dict) else '?'})")
else:
    print('No checkpoint found; evaluating current model weights.')

test_acc = evaluate(test_loader, model, desc='Testing')
print('=' * 48)
print(f'FINAL TEST ACCURACY ({TEST_WAY}-way {SHOT}-shot): {test_acc:.2f}%')
print('=' * 48)
print('Learned scale weights:', [round(v, 4) for v in model.stage_weights.tolist()])
print('Checkpoint:', best_checkpoint_path)

Loaded best checkpoint (epoch 1)


Testing:   0%|          | 0/600 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed304778ea0>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed304778ea0>Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
    self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():
      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
if w.is_alive():    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.13/multiprocessing/process.py", line 16

FINAL TEST ACCURACY (5-way 5-shot): 82.97%
Learned scale weights: [0.3358, 0.4047, 0.5177, 0.7866, 1.8535]
Checkpoint: /kaggle/working/saved_models/best_model_5shot.pth


In [11]:
print('gamma1:', model.gamma1_l)
print('gamma2:', model.gamma2_l)

gamma1: Parameter containing:
tensor(0.2000, device='cuda:0')
gamma2: Parameter containing:
tensor(0.2000, device='cuda:0')
